# 1) Resume Parser → processed/resumes.json

In [ ]:
%pip install pdfplumber python-docx chardet tqdm

In [ ]:

from pathlib import Path
import json, re, chardet
from tqdm import tqdm
import pdfplumber

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

def read_txt(path: Path) -> str:
    raw = path.read_bytes()
    enc = chardet.detect(raw).get("encoding") or "utf-8"
    return raw.decode(enc, errors="ignore")

def read_pdf(path: Path) -> str:
    text = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text.append(page.extract_text() or "")
    return "\n".join(text)

def read_docx(path: Path) -> str:
    from docx import Document
    d = Document(str(path))
    return "\n".join(p.text for p in d.paragraphs)

def clean_text(s: str) -> str:
    import re
    s = s.replace("\u00a0", " ")
    s = re.sub(r"[\t\r]", " ", s)
    s = re.sub(r" +", " ", s)
    s = re.sub(r"\n\s*\n+", "\n", s)
    return s.strip()

def extract_text(path: Path) -> str:
    ext = path.suffix.lower()
    if ext == ".txt": return clean_text(read_txt(path))
    if ext == ".pdf": return clean_text(read_pdf(path))
    if ext == ".docx": return clean_text(read_docx(path))
    return ""

files = sorted([p for p in RAW.glob("**/*") if p.suffix.lower() in {".txt",".pdf",".docx"}])
print(f"Found {len(files)} files")

resumes = []
for f in tqdm(files, desc="Processing"):
    text = extract_text(f)
    if text:
        resumes.append({"id": f.stem, "source": str(f.relative_to(RAW)), "text": text})

out_path = PROCESSED / "resumes.json"
out_path.write_text(json.dumps(resumes, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Saved {len(resumes)} → {out_path}")
